# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The dataset structure is defined by record sets, which group fields (columns) together. Here, we'll enumerate the available record sets and the fields/columns within each, using their `@id` for precise referencing.

In [ ]:
# List available Record Sets by @id and preview their fields/columns
record_sets = metadata.record_sets
print(f"Found {len(record_sets)} record sets:")
for rs in record_sets:
    print(f"\nRecord Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields/Columns:")
        for f in rs.fields:
            print(f"    - Field name: {f.name}")
            print(f"      @id: {f.id}")
            if getattr(f, 'data_type', None):
                print(f"      dataType: {f.data_type}")
    else:
        print("  No fields listed.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We'll demonstrate extracting all data from all available record sets, referencing each via their `@id`, and store the data in pandas DataFrames.

In [ ]:
# Collect all record set @id values
record_set_ids = [rs.id for rs in metadata.record_sets]

# Load data from each record set into a DataFrame
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for Record Set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Sample data:\n{df.head(2)}\n")

# For demonstration, select the first available record set to proceed with EDA
if record_set_ids:
    selected_record_set_id = record_set_ids[0]
    print(f"Selected Record Set for EDA: {selected_record_set_id}")
else:
    selected_record_set_id = None
    print("No record sets available.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

We'll select a numeric field and a group-by field from the selected record set by referencing their `@id`s.

In [ ]:
if selected_record_set_id is None or dataframes[selected_record_set_id].empty:
    print("No data available for EDA.")
else:
    df = dataframes[selected_record_set_id]

    # Attempt to select a numeric field for analysis based on detected types or metadata
    # If schema provides data types, use; otherwise, guess from DataFrame
    
    # Get fields for selected record set
    fields = None
    for rs in metadata.record_sets:
        if rs.id == selected_record_set_id:
            fields = getattr(rs, 'fields', None)
            break

    # Find first numeric field (float or integer)
    numeric_field_id = None
    group_field_id = None
    
    if fields is not None:
        for field in fields:
            # Check data type from schema
            dtype = getattr(field, 'data_type', None)
            if dtype in ('Float', 'Number', 'Integer', 'schema:Float', 'schema:Number', 'schema:Integer'):
                numeric_field_id = field.id
                break
    
    # If not found in schema, try from pandas
    if numeric_field_id is None:
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break

    if fields is not None:
        for field in fields:
            # group by first categorical/text field (skip the numeric column)
            dtype = getattr(field, 'data_type', None)
            if field.id != numeric_field_id and (dtype is None or dtype in ('Text', 'schema:Text', 'String', 'schema:String')):
                group_field_id = field.id
                break
    if group_field_id is None:
        # Try to find a column that is object type in pandas
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == 'object':
                group_field_id = col
                break

    print(f"Numeric field selected: {numeric_field_id}")
    print(f"Group-by field selected: {group_field_id}")

    # Ensure the numeric column is converted for analysis
    if numeric_field_id is not None and numeric_field_id in df.columns:
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

        # Example: Filter records with numeric_field > threshold
        threshold = df[numeric_field_id].quantile(0.9) if df[numeric_field_id].notnull().any() else 0
        filtered_df = df[df[numeric_field_id] > threshold]

        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalization/z-score
        if not filtered_df.empty:
            filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group and aggregate
        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No suitable numeric field found for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We will use the selected numeric and group fields (referenced by their `@id`) to plot example distributions.

In [ ]:
if selected_record_set_id is None or dataframes[selected_record_set_id].empty:
    print("No data available for visualization.")
else:
    df = dataframes[selected_record_set_id]
    # Ensure types
    if numeric_field_id is not None and numeric_field_id in df.columns:
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
        plt.title(f'Distribution of field: {numeric_field_id}')
        plt.xlabel(numeric_field_id)
        plt.ylabel('Count')
        plt.show()
        
        # Grouped boxplot
        if group_field_id is not None and group_field_id in df.columns:
            plt.figure(figsize=(10,5))
            sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
            plt.title(f'{numeric_field_id} by {group_field_id}')
            plt.xlabel(group_field_id)
            plt.ylabel(numeric_field_id)
            plt.xticks(rotation=45)
            plt.show()
        else:
            print("No suitable group field found for boxplot.")
    else:
        print("No suitable numeric field found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we demonstrated how to load, inspect, and analyze data from a Croissant schema-defined dataset using the `mlcroissant` library. All references to record sets and fields use the schema's canonical `@id` identifiers for reproducibility. Continue your EDA or modeling work based on the normalized and filtered DataFrames above.